# fase_1 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 1.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import pickle
import json


## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_future'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to target database (db_future config): {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()

# Mengambil nama tabel dari hasil query
# Note: Format output 'SHOW TABLES' biasanya {'Tables_in_dbname': 'tablename'}
target_tables = [list(t.values())[0] for t in tables_data]

print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")
print(target_tables)

# Dictionary untuk menyimpan data yang sudah di-load
data_frames = {}

print("\n--- Memulai proses load semua data tabel ---")

for table in target_tables:
    try:
        # Load data menggunakan pandas langsung dari koneksi SQL
        query = f"SELECT * FROM `{table}`"
        data_frames[table] = pd.read_sql(query, db_old)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(data_frames[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'data_frames' ---")
print("Kamu sekarang bisa akses datanya dengan: data_frames['nama_tabel']")

# Contoh akses data:
# print(data_frames['users'].head())

# Tutup koneksi jika sudah tidak digunakan
# db_old.close()
# db_new.close()


--- Ditemukan 108 tabel di Database Lama ---
['absensi', 'absensi_note', 'bidang', 'bidangkategori', 'bidanglink', 'calon', 'calon_detil', 'calon_pertanyaan', 'calon_pertanyaan_detil', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_siswa', 'catatan_siswa_follow_up', 'catatanawal_admin', 'catatanawal_datautama', 'catatanawal_infolain', 'catatanawal_tglpenting', 'divisi', 'docs', 'file_rapor_siswa', 'form', 'form_calon', 'form_calon_detil1', 'form_calon_detil2', 'form_calon_detil3', 'form_calon_detil4', 'format_rapor', 'format_rapor_detil', 'format_rapor_detil_rumus', 'format_rapor_rumus', 'format_raport_level', 'hakakses', 'histori_pengajuan', 'history_rapor', 'identitas', 'infrastruktur', 'jabatan', 'jadwal', 'jadwal_detil', 'jadwal_pengajar', 'jadwal_siswa', 'jamkerja', 'kabupaten', 'karyawan', 'kecamatan', 'keluar', 'keluarga', 'kelurahan', 'kurikulum', 'kurikulum_detil', 'kurikulum_detil_sub', 'kurikulum_kelas', 'kursus', 'leapprofil', 'leapverse', 'level', 'lib

In [4]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS (DATABASE BARU)
# ---------------------------------------------------------
# Menggunakan cursor dari database baru
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()

# Mengambil nama tabel dari hasil query
target_tables_new = [list(t.values())[0] for t in tables_data_new]

print(f"\n--- Ditemukan {len(target_tables_new)} tabel di Database Baru ---")
print(target_tables_new)

# Dictionary untuk menyimpan data dari database baru (jika diperlukan untuk verifikasi)
data_frames_new = {}

print("\n--- Memulai proses load semua data dari Database Baru ---")

for table in target_tables_new:
    try:
        # Load data menggunakan pandas dengan koneksi database baru
        query = f"SELECT * FROM `{table}`"
        data_frames_new[table] = pd.read_sql(query, db_new)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(data_frames_new[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table} dari DB Baru: {e}")

print("\n--- Proses load selesai. Data DB Baru tersimpan di 'data_frames_new' ---")


--- Ditemukan 113 tabel di Database Baru ---
['absensi', 'activity_log', 'admin_sarpras', 'bidang_kategori', 'bidang_link', 'busdev_bidang', 'cache', 'cache_locks', 'calon_siswa', 'calon_siswa_akademik', 'calon_siswa_bayar', 'calon_siswa_fo_detail', 'calon_siswa_form_program_requirements', 'calon_siswa_form_programs', 'calon_siswa_jadwal', 'calon_siswa_kursus', 'calon_siswa_ortu', 'calon_siswa_proses', 'calon_siswa_proses_logs', 'calon_siswa_status_logs', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_remidi_siswa', 'catatan_siswa', 'division_user', 'divisions', 'failed_jobs', 'followup_cs', 'histori_pengajuan', 'izin_karyawan', 'jadwal', 'jadwal_detail', 'jadwal_detail_logs', 'jadwal_hari', 'jadwal_pengajar', 'jadwal_siswa', 'job_batches', 'jobs', 'kabupaten', 'karyawan', 'karyawan_resign', 'kecamatan', 'keluarga_karyawan', 'kelurahan', 'kemitraan_verifikator', 'kontak_prospek', 'kursus', 'kursus_level', 'kursus_libur', 'kursus_siswa', 'level', 'libur', 'log_aktiv

In [5]:
# Mapping tabel bagian Hanif: (Tabel Lama, Tabel Baru)
hanif_tables_map = [
    ('role', 'roles'),
    ('bidang', 'busdev_bidang'),
    ('syarat', 'syarat_resign'),
    ('ttd', 'ttd'),
    ('tag_keluar', 'tag_siswa_keluar')
]

# Tempat menyimpan data mentah dan metadata skema
raw_data = {}

print("=== INSPEKSI SKEMA DETAIL & RETRIEVAL DATA (SEMUA TABEL) ===\n")

for old_t, new_t in hanif_tables_map:
    try:
        print(f"📦 ANALISIS: {old_t} ➔ {new_t}")
        
        cursor_old.execute(f"DESCRIBE {old_t}")
        df_old_schema = pd.DataFrame(cursor_old.fetchall())

        cursor_new.execute(f"DESCRIBE {new_t}")
        df_new_schema = pd.DataFrame(cursor_new.fetchall())

        cursor_old.execute(f"SELECT * FROM {old_t}")
        raw_data[old_t] = cursor_old.fetchall()

        display(df_old_schema)
        display(df_new_schema)

        print(f"✅ {len(raw_data[old_t])} records\n")

    except Exception as e:
        print(f"❌ ERROR di tabel {old_t}: {e}")

print("✓ Semua metadata skema dan data mentah 5 tabel Hanif berhasil dimuat.")
print(f"\n===== {old_t} DONE =====\n")

=== INSPEKSI SKEMA DETAIL & RETRIEVAL DATA (SEMUA TABEL) ===

📦 ANALISIS: role ➔ roles


,Field,Type,Null,Key,Default,Extra
0,idrole,varchar(6),NO,PRI,,
1,nama_role,varchar(45),NO,,,


,Field,Type,Null,Key,Default,Extra
0,id,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_division,bigint(20) unsigned,YES,MUL,None,
2,name,varchar(255),NO,,None,
3,guard_name,varchar(255),NO,,None,
4,created_at,timestamp,YES,,None,
5,updated_at,timestamp,YES,,None,


✅ 9 records

📦 ANALISIS: bidang ➔ busdev_bidang


,Field,Type,Null,Key,Default,Extra
0,idbidang,int(10) unsigned,NO,PRI,None,auto_increment
1,namabidang,varchar(250),NO,,,


...

,Field,Type,Null,Key,Default,Extra
0,id_ttd,bigint(20) unsigned,NO,PRI,None,auto_increment
1,ttd,text,NO,,None,
2,status,varchar(50),NO,,None,


✅ 1 records

📦 ANALISIS: tag_keluar ➔ tag_siswa_keluar


,Field,Type,Null,Key,Default,Extra
0,idtag,varchar(6),NO,PRI,,
1,tag,varchar(250),NO,,,
2,keterangan,text,YES,,None,


,Field,Type,Null,Key,Default,Extra
0,id_tag_keluar,bigint(20) unsigned,NO,PRI,None,auto_increment
1,nama_tag,varchar(100),NO,,None,
2,keterangan_keluar,text,NO,,None,


✅ 11 records

✓ Semua metadata skema dan data mentah 5 tabel Hanif berhasil dimuat.

===== tag_keluar DONE =====



## 3. Transform Data (jika diperlukan)

In [6]:
from datetime import datetime
now = datetime.now()
transformed_dfs = {}

# 1. roles
# Ambil ID dan Nama, tambahkan guard_name dan timestamps
df_roles = pd.DataFrame(raw_data['role'])
df_roles = df_roles.rename(columns={'idrole': 'id', 'nama_role': 'name'})
df_roles['name'] = df_roles['name'].astype(str).str.strip()
# Ekstrak angka dari 'R00001' karena target kolom 'id' adalah bigint
df_roles['id'] = df_roles['id'].str.extract('(\d+)').astype(int)
df_roles['guard_name'] = 'web'
df_roles['created_at'] = now
df_roles['updated_at'] = now
transformed_dfs['roles'] = df_roles[['id', 'name', 'guard_name', 'created_at', 'updated_at']]

# 2. busdev_bidang
# Ambil ID dan Nama Bidang
df_bidang = pd.DataFrame(raw_data['bidang'])
df_bidang = df_bidang.rename(columns={'idbidang': 'id_bidang', 'namabidang': 'nama_bidang'})
df_bidang['nama_bidang'] = df_bidang['nama_bidang'].astype(str).str.strip()
transformed_dfs['busdev_bidang'] = df_bidang[['id_bidang', 'nama_bidang']]

# 3. syarat_resign
# Ambil ID dan Isi Syarat
df_syarat = pd.DataFrame(raw_data['syarat'])
df_syarat = df_syarat.rename(columns={'idsyarat': 'id_syarat', 'syarat': 'isi_syarat'})
df_syarat['isi_syarat'] = df_syarat['isi_syarat'].astype(str)
transformed_dfs['syarat_resign'] = df_syarat[['id_syarat', 'isi_syarat']]

# 4. ttd
# Ambil ID, ttd dan status
df_ttd = pd.DataFrame(raw_data['ttd'])
df_ttd = df_ttd.rename(columns={'idttd': 'id_ttd'})
# Ekstrak angka dari 'T00001' karena target kolom 'id_ttd' adalah bigint
df_ttd['id_ttd'] = df_ttd['id_ttd'].str.extract('(\d+)').astype(int)
df_ttd['ttd'] = df_ttd['ttd'].astype(str)
df_ttd['status'] = df_ttd['status'].astype(str)
transformed_dfs['ttd'] = df_ttd[['id_ttd', 'ttd', 'status']]

# 5. tag_siswa_keluar
# Ambil ID, Nama Tag dan Keterangan
df_tag = pd.DataFrame(raw_data['tag_keluar'])
df_tag = df_tag.rename(columns={'idtag': 'id_tag_keluar', 'tag': 'nama_tag', 'keterangan': 'keterangan_keluar'})
# Ekstrak angka dari 'TK0001' karena target kolom 'id_tag_keluar' adalah bigint
df_tag['id_tag_keluar'] = df_tag['id_tag_keluar'].str.extract('(\d+)').astype(int)
df_tag['nama_tag'] = df_tag['nama_tag'].astype(str)
df_tag['keterangan_keluar'] = df_tag['keterangan_keluar'].fillna('').astype(str)
transformed_dfs['tag_siswa_keluar'] = df_tag[['id_tag_keluar', 'nama_tag', 'keterangan_keluar']]

print("✓ Transformasi selesai. Kolom ID disertakan untuk menjaga relasi (Foreign Key).")

✓ Transformasi selesai. Kolom ID disertakan untuk menjaga relasi (Foreign Key).


In [7]:
# Tabel ekspektasi dari DATABASE_SCHEMA.md
expected_counts = {
    'roles': 9,
    'busdev_bidang': 4,
    'syarat_resign': 1,
    'ttd': 1,
    'tag_siswa_keluar': 11
}

print("=== VERIFIKASI HASIL TRANSFORMASI ===")
for table_name, df in transformed_dfs.items():
    actual_count = len(df)
    expected = expected_counts.get(table_name)
    status = "✓ OK" if actual_count == expected else f"⚠ MISMATCH (Ekspektasi: {expected})"
    
    print(f"\nTabel: {table_name} ({status})")
    print(f"Columns: {df.columns.tolist()}")
    # Menampilkan 2 baris teratas untuk cek data
    display(df.head(2))



=== VERIFIKASI HASIL TRANSFORMASI ===

Tabel: roles (✓ OK)
Columns: ['id', 'name', 'guard_name', 'created_at', 'updated_at']


,id,name,guard_name,created_at,updated_at
0,1,HR,web,2026-06-21 18:50:43.302833,2026-06-21 18:50:43.302833
1,2,KARYAWAN,web,2026-06-21 18:50:43.302833,2026-06-21 18:50:43.302833



Tabel: busdev_bidang (✓ OK)
Columns: ['id_bidang', 'nama_bidang']


,id_bidang,nama_bidang
0,7,Sales & Marketing
1,8,R & D



Tabel: syarat_resign (✓ OK)
Columns: ['id_syarat', 'isi_syarat']


,id_syarat,isi_syarat
0,1,<p>SOP pemutusan kerja dan pengunduran diri Ka...



Tabel: ttd (✓ OK)
Columns: ['id_ttd', 'ttd', 'status']


,id_ttd,ttd,status
0,1,1775719060_815664e38e280d4e971a.png,Ya



Tabel: tag_siswa_keluar (✓ OK)
Columns: ['id_tag_keluar', 'nama_tag', 'keterangan_keluar']


,id_tag_keluar,nama_tag,keterangan_keluar
0,1,AKADEMIK,"Jika siswa tidak naik/lulus level, merasa tida..."
1,2,APLIKASI,Jika siswa merasa kesulitan mengoperasikan Lea...


## 4. Simpan ke Pickle untuk Insert Handler

In [8]:
# Simpan ke file .pkl untuk diproses oleh insert_handler.ipynb
file_name = 'fase_1_hanif.pkl'
import pickle

try:
    with open(file_name, 'wb') as f:
        pickle.dump(transformed_dfs, f)
    print(f"✅ Berhasil menyimpan {len(transformed_dfs)} tabel ke {file_name}")
    print("Siap diproses oleh insert_handler.ipynb!")
except Exception as e:
    print(f"❌ Gagal simpan pickle: {e}")

✅ Berhasil menyimpan 5 tabel ke fase_1_hanif.pkl
Siap diproses oleh insert_handler.ipynb!


## 5. Verifikasi Data (Lokal)

In [9]:
import json
from datetime import datetime
import pickle

print("=== VERIFIKASI DATA FASE 1 HANIF SEBELUM EXPORT ===\n")

# Load kembali untuk verifikasi file pkl
try:
    with open('fase_1_hanif.pkl', 'rb') as f:
        loaded_data = pickle.load(f)
    print(f"✓ Berhasil me-load kembali 'fase_1_hanif.pkl' untuk verifikasi.\n")
except Exception as e:
    print(f"❌ Gagal load pkl untuk verifikasi: {e}")
    loaded_data = {}

migration_summary = []
all_verified = True

# Kita gunakan mapping yang sama dari Step 2
for old_t, new_t in hanif_tables_map:
    try:
        # 1. Ambil jumlah record dari df hasil transform
        df = loaded_data.get(new_t, pd.DataFrame())
        count_new = len(df)
        
        # 2. Ambil jumlah record dari DB Lama
        count_old = len(raw_data[old_t])
        
        is_match = (count_new == count_old)
        if not is_match:
            all_verified = False
            
        print(f"📊 Tabel: {new_t}")
        print(f"   - Sumber ({old_t}): {count_old} baris")
        print(f"   - Hasil Transformasi: {count_new} baris")
        print(f"   - Status: {'✓ MATCH' if is_match else '⚠ MISMATCH'}")
        
        migration_summary.append({
            'tabel_baru': new_t,
            'tabel_lama': old_t,
            'records_old': count_old,
            'records_new': count_new,
            'status': 'MATCH' if is_match else 'MISMATCH'
        })
        
    except Exception as e:
        print(f"   ❌ Gagal verifikasi tabel {new_t}: {e}")
        all_verified = False
    print("-" * 45)

total_old = sum(item['records_old'] for item in migration_summary)
total_new = sum(item['records_new'] for item in migration_summary)

migration_result = {
    'fase': 'fase_1',
    'script': 'script_hanif',
    'fase_num': 1,
    'status': 'ready_for_insert' if all_verified else 'warning',
    'records_transformed': total_new,
    'pickle_file': 'fase_1_hanif.pkl',
    'verified': all_verified,
    'timestamp': datetime.now().isoformat(),
    'details': migration_summary,
    'message': f"Transformasi 5 tabel master Hanif selesai dengan status {'Aman' if all_verified else 'Perlu Cek'}"
}

print("\n" + "="*60)
print("HASIL AKHIR TRANSFORMASI - FASE 1 / SCRIPT_HANIF")
print("="*60)
print(json.dumps(migration_result, indent=2))
print("="*60)

=== VERIFIKASI DATA FASE 1 HANIF SEBELUM EXPORT ===

✓ Berhasil me-load kembali 'fase_1_hanif.pkl' untuk verifikasi.

📊 Tabel: roles
   - Sumber (role): 9 baris
   - Hasil Transformasi: 9 baris
   - Status: ✓ MATCH
---------------------------------------------
📊 Tabel: busdev_bidang
   - Sumber (bidang): 4 baris
   - Hasil Transformasi: 4 baris
   - Status: ✓ MATCH
---------------------------------------------
📊 Tabel: syarat_resign
   - Sumber (syarat): 1 baris
   - Hasil Transformasi: 1 baris
   - Status: ✓ MATCH
---------------------------------------------
📊 Tabel: ttd
   - Sumber (ttd): 1 baris
   - Hasil Transformasi: 1 baris
   - Status: ✓ MATCH
---------------------------------------------
📊 Tabel: tag_siswa_keluar
   - Sumber (tag_keluar): 11 baris
   - Hasil Transformasi: 11 baris
   - Status: ✓ MATCH
---------------------------------------------

HASIL AKHIR TRANSFORMASI - FASE 1 / SCRIPT_HANIF
{
  "fase": "fase_1",
  "script": "script_hanif",
  "fase_num": 1,
  "status": "r

## Close Connection

In [10]:
# Close semua koneksi database
try:
    cursor_old.close()
    cursor_new.close()
    db_old.close()
    db_new.close()
    print("✓ Database connections closed")
except:
    print("⚠ Error closing connections (mungkin sudah tertutup)")

✓ Database connections closed
